# Self-RAG：检索增强生成的动态方法

## 概述

Self-RAG 是一种先进的算法，结合了自然语言处理中基于检索和基于生成的方法的强大功能。它动态地决定是否使用检索到的信息以及如何最好地利用它来生成响应，旨在产生更准确、相关和有用的输出。

## 动机

传统的问答系统常常难以平衡检索信息的使用和新内容的生成。一些系统可能过于依赖检索到的数据，导致响应缺乏灵活性，而其他系统可能会在没有充分事实信息基础的情况下生成响应。 Self-RAG 通过实施多步骤流程来解决这些问题，该流程仔细评估检索到的信息的必要性和相关性，并评估生成的响应的质量。

## 关键组件

1. **检索决策**：确定给定查询是否需要检索。
2. **文档检索**：从支持存储中获取潜在的相关文档。
3. **相关性评估**：评估检索到的文档与查询的相关性。
4. **响应生成**：根据相关上下文生成响应。
5. **支持评估**：评估上下文支持生成的响应的程度。
6. **效用评估**：对生成的响应的有用性进行评级。

## 方法详细信息

1. **检索决策**：算法首先决定给定查询是否需要检索。此步骤可防止对可直接回答的查询进行不必要的检索。

2. **文档检索**：如果认为有必要进行检索，则算法会从支持存储中获取前 k 个最相似的文档。

3. **相关性评估**：评估每个检索到的文档与查询的相关性。此步骤过滤掉不相关的信息，确保仅使用相关的上下文进行生成。

4. **响应生成**：算法使用相关上下文生成响应。如果没有找到相关上下文，它会生成响应而不进行检索。

5. **支持评估**：对每个生成的响应进行评估，以确定上下文对它的支持程度。此步骤有助于识别基于所提供信息的响应。

6. **效用评估**：考虑每个响应解决原始查询的效果，对每个响应的效用进行评级。

7. **响应选择**：最后一步涉及根据支持评估和效用评估选择最佳响应。

## 该方法的优点

1. **动态检索**：通过判断是否需要检索，系统可以高效地适应不同类型的查询。

2. **相关性过滤**：相关性评估步骤确保仅使用相关信息，减少生成过程中的噪音。

3. **质量保证**：支持评估和效用评估提供了一种衡量生成响应质量的方法。

4. **灵活性**：系统可以在检索或不检索的情况下生成响应，以适应可用信息。

5. **提高准确性**：通过将响应建立在相关检索信息中并评估其支持，系统可以产生更准确的输出。

## 结论

Self-RAG 代表了一种复杂的问答和信息检索任务方法。通过合并多个评估步骤并动态决定检索信息的使用，它的目标是生成不仅相关且准确而且对最终用户有用的响应。这种方法展示了以深思熟虑、评估的方式结合检索和生成技术的潜力，以提高人工智能生成的响应的质量。

<div style="text-align: center;">

<img src="../images/self_rag.svg" alt="Self RAG" style="width:80%; height:auto;">
</div>

# 包安装和导入

下面的单元格安装运行此笔记本所需的所有必需软件包。

In [ ]:
# Install required packages
!pip install langchain langchain-openai python-dotenv

In [ ]:
# Clone the repository to access helper functions and evaluation modules
!git clone https://github.com/NirDiamant/RAG_TECHNIQUES.git
import sys
sys.path.append('RAG_TECHNIQUES')
# If you need to run with the latest data
# !cp -r RAG_TECHNIQUES/data .

In [3]:
import os
import sys
from dotenv import load_dotenv
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.pydantic_v1 import BaseModel, Field


# Original path append replaced for Colab compatibility
from helper_functions import *
from evaluation.evalute_rag import *

# Load environment variables from a .env file
load_dotenv()

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_API_KEY')

### 定义文件路径

In [ ]:
# Download required data files
import os
os.makedirs('data', exist_ok=True)

# Download the PDF document used in this notebook
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf
!wget -O data/Understanding_Climate_Change.pdf https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/Understanding_Climate_Change.pdf


In [4]:
path = "data/Understanding_Climate_Change.pdf"

### 创建一个支持存储

In [5]:
vectorstore = encode_pdf(path)

### 初始化语言模型

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", max_tokens=1000, temperature=0)

### 定义提示模板

In [8]:
class RetrievalResponse(BaseModel):
    response: str = Field(..., title="Determines if retrieval is necessary", description="Output only 'Yes' or 'No'.")
retrieval_prompt = PromptTemplate(
    input_variables=["query"],
    template="Given the query '{query}', determine if retrieval is necessary. Output only 'Yes' or 'No'."
)

class RelevanceResponse(BaseModel):
    response: str = Field(..., title="Determines if context is relevant", description="Output only 'Relevant' or 'Irrelevant'.")
relevance_prompt = PromptTemplate(
    input_variables=["query", "context"],
    template="Given the query '{query}' and the context '{context}', determine if the context is relevant. Output only 'Relevant' or 'Irrelevant'."
)

class GenerationResponse(BaseModel):
    response: str = Field(..., title="Generated response", description="The generated response.")
generation_prompt = PromptTemplate(
    input_variables=["query", "context"],
    template="Given the query '{query}' and the context '{context}', generate a response."
)

class SupportResponse(BaseModel):
    response: str = Field(..., title="Determines if response is supported", description="Output 'Fully supported', 'Partially supported', or 'No support'.")
support_prompt = PromptTemplate(
    input_variables=["response", "context"],
    template="Given the response '{response}' and the context '{context}', determine if the response is supported by the context. Output 'Fully supported', 'Partially supported', or 'No support'."
)

class UtilityResponse(BaseModel):
    response: int = Field(..., title="Utility rating", description="Rate the utility of the response from 1 to 5.")
utility_prompt = PromptTemplate(
    input_variables=["query", "response"],
    template="Given the query '{query}' and the response '{response}', rate the utility of the response from 1 to 5."
)

# Create LLMChains for each step
retrieval_chain = retrieval_prompt | llm.with_structured_output(RetrievalResponse)
relevance_chain = relevance_prompt | llm.with_structured_output(RelevanceResponse)
generation_chain = generation_prompt | llm.with_structured_output(GenerationResponse)
support_chain = support_prompt | llm.with_structured_output(SupportResponse)
utility_chain = utility_prompt | llm.with_structured_output(UtilityResponse)

### 定义自RAG逻辑流程

In [11]:
def self_rag(query, vectorstore, top_k=3):
    print(f"\nProcessing query: {query}")
    
    # Step 1: Determine if retrieval is necessary
    print("Step 1: Determining if retrieval is necessary...")
    input_data = {"query": query}
    retrieval_decision = retrieval_chain.invoke(input_data).response.strip().lower()
    print(f"Retrieval decision: {retrieval_decision}")
    
    if retrieval_decision == 'yes':
        # Step 2: Retrieve relevant documents
        print("Step 2: Retrieving relevant documents...")
        docs = vectorstore.similarity_search(query, k=top_k)
        contexts = [doc.page_content for doc in docs]
        print(f"Retrieved {len(contexts)} documents")
        
        # Step 3: Evaluate relevance of retrieved documents
        print("Step 3: Evaluating relevance of retrieved documents...")
        relevant_contexts = []
        for i, context in enumerate(contexts):
            input_data = {"query": query, "context": context}
            relevance = relevance_chain.invoke(input_data).response.strip().lower()
            print(f"Document {i+1} relevance: {relevance}")
            if relevance == 'relevant':
                relevant_contexts.append(context)
        
        print(f"Number of relevant contexts: {len(relevant_contexts)}")
        
        # If no relevant contexts found, generate without retrieval
        if not relevant_contexts:
            print("No relevant contexts found. Generating without retrieval...")
            input_data = {"query": query, "context": "No relevant context found."}
            return generation_chain.invoke(input_data).response
        
        # Step 4: Generate response using relevant contexts
        print("Step 4: Generating responses using relevant contexts...")
        responses = []
        for i, context in enumerate(relevant_contexts):
            print(f"Generating response for context {i+1}...")
            input_data = {"query": query, "context": context}
            response = generation_chain.invoke(input_data).response
            
            # Step 5: Assess support
            print(f"Step 5: Assessing support for response {i+1}...")
            input_data = {"response": response, "context": context}
            support = support_chain.invoke(input_data).response.strip().lower()
            print(f"Support assessment: {support}")
            
            # Step 6: Evaluate utility
            print(f"Step 6: Evaluating utility for response {i+1}...")
            input_data = {"query": query, "response": response}
            utility = int(utility_chain.invoke(input_data).response)
            print(f"Utility score: {utility}")
            
            responses.append((response, support, utility))
        
        # Select the best response based on support and utility
        print("Selecting the best response...")
        best_response = max(responses, key=lambda x: (x[1] == 'fully supported', x[2]))
        print(f"Best response support: {best_response[1]}, utility: {best_response[2]}")
        return best_response[0]
    else:
        # Generate without retrieval
        print("Generating without retrieval...")
        input_data = {"query": query, "context": "No retrieval necessary."}
        return generation_chain.invoke(input_data).response

### 测试自RAG功能轻松查询关联度高

In [ ]:
query = "What is the impact of climate change on the environment?"
response = self_rag(query, vectorstore)

print("\nFinal response:")
print(response)

### 使用更具挑战性且相关性较低的查询来测试 self-RAG 函数

In [ ]:
query = "how did harry beat quirrell?"
response = self_rag(query, vectorstore)

print("\nFinal response:")
print(response)

![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--self-rag)